# Spectral Normalization

Created by Sharon Xuesong Wang of THU ObsAstro, 2022.4.26 \
Updated for ObsAstro25, 2025.4.23 \
Updated for ObsAstro26, 2026.4.29

## Objective

Extract a 1-D spectrum from a 2-D image of a long-slit spectrum, determine wavelength solution, and then measure the redshift of the target.


## Key steps
- Define target and background apertures
- Subtract the background contribution from the target aperture
- Extract (sum) the counts in the background subtracted target aperture
- Load in the wavelength solution
- Plot the extracted 1-D spectrum


## Required dependencies

Please check that you can import the following python packages without any problem. If anyone is missing (likely specutils), please use `pip install <module>` in your terminal **or** the code block below to install it.

* python 3
* astropy
* numpy
* scipy
* matplotlib
* specutils

In [ ]:
# Install the missing python package - You need internet to do this!
import sys
!{sys.executable} -m pip install specutils

# Try Conda if pip doesn't work
# !conda install --yes --prefix {sys.prefix} specutils

In [ ]:
# python modules that we will use
import os
import numpy as np
from astropy.io import fits
from astropy.stats import sigma_clip
from scipy.optimize import curve_fit
from scipy.signal import find_peaks
from pathlib import Path
import specutils

%matplotlib inline
import matplotlib.pylab as plt

In [ ]:
# change plotting defaults
plt.rc('axes', labelsize=14)
plt.rc('axes', labelweight='bold')
plt.rc('axes', titlesize=16)
plt.rc('axes', titleweight='bold')
plt.rc('font', family='sans-serif')
plt.rcParams['figure.figsize'] = (17, 7)

## Read in the solar spectrum

In [ ]:
# load the wavelength calibrated spectrum
from astropy.io import ascii

target_name = 'SN2026lmp'
filedir = f'../data/{target_name}'
filepath = list(Path(filedir).glob('*.txt'))
spec_data = ascii.read(filepath[0])
wavs = spec_data['col1']
spec = spec_data['col2']
try:
    norm_spec = spec_data['col3']  # this is the continuum normalized spectrum, i.e., the right answer! 
                               # We don't need this just yet
except KeyError:
    norm_spec = None

In [ ]:
# plot the spectrum here
plt.figure()
plt.plot(wavs, spec, color='k', lw=0.5)
plt.title(f'The {target_name} spectrum')
plt.xlabel('Wavelength (Angstroms)')
plt.ylabel('Flux')
plt.show()

## Do a quick continuum fit

In [ ]:
# now let's try to remove the continuum
# First, we need some functions from the specutils package
from specutils.spectra import Spectrum1D, SpectralRegion
from specutils.fitting import fit_generic_continuum
from astropy import units as u

In [ ]:
# This package needs to define a Spectrum1D object, which must have units
# The unit for the flux is counts, and it's angstrom for the wavelength.
spectrum = Spectrum1D(flux=spec*u.ct, spectral_axis=wavs*u.angstrom)

In [ ]:
# Let's use the fit continuum function to do a quick fit
g1_fit = fit_generic_continuum(spectrum)
cont_fitted = g1_fit(wavs*u.angstrom)

In [ ]:
# plot your spectrum and the fitted continuum in dash lines here
plt.figure()
plt.plot(wavs, spec, color='k', lw=0.5,label='Original Spectrum')  # original spectrum
plt.plot(wavs, cont_fitted, '--', color='r', lw=0.5,label='Fitted Continuum')  # the continuum you are fitting
plt.title('Spectrum and Fitted Continuum')
plt.xlabel('Wavelength (Angstroms)')
plt.ylabel('Flux')
plt.legend()
plt.show()

## Improve the fit by tweeking some parameters

There are 2 key parameters we can change in fitting the continuum:

* median_window\
The width for the window to apply a median filter to the spectrum, the default is only 3.

* model\
The model we would like to use to fit the continuum shape, we use the Chebyshev polynomials, which are quite flexible. The default is a 3rd order Chebyshev polynomial. You can change this to a higher order one.

In [ ]:
from astropy.modeling.models import Chebyshev1D

# The parameters are at their default values now. Change them to see if you can get a better fit.
g1_fit = fit_generic_continuum(spectrum, median_window=3, model=Chebyshev1D(3))
cont_fitted = g1_fit(wavs*u.angstrom)

In [ ]:
# plot your fit here to check it
plt.figure()
plt.plot(wavs, spec, color='k', lw=0.5, label='Original Spectrum')
plt.plot(wavs, cont_fitted, '--', color='r', lw=0.5, label='Fitted Continuum')
plt.xlabel('Wavelength (Angstroms)')
plt.ylabel('Flux')
plt.title('Spectrum and Fitted Continuum')
plt.legend()
plt.show()

## Exclude prominent lines for a better continuum fit

In [ ]:
# Let's do a better one by defining good spectral regions free of strong lines.
# This requires another function, fit_continuum
from specutils.fitting.continuum import fit_continuum

# Define the "continuum regions" here by entering the appropriate wavelengths replacing the ???.
# This example here only excludes 1 strong line.
# You can add another region to exclude the second strong line in this spectrum to see if you will do better.
# region = [(np.min(wavs)*u.angstrom,4110*u.angstrom),\
#           (4112*u.angstrom,4117*u.angstrom),\
#           (4118.5*u.angstrom,4124*u.angstrom),\
#           (4125*u.angstrom,4127*u.angstrom),\
#           (4131*u.angstrom,4133.5*u.angstrom),\
#           (4134.5*u.angstrom,np.max(wavs)*u.angstrom)]
region = [(4500*u.angstrom, np.max(wavs)*u.angstrom)]

# Fit your designated spectral region with a continuum
fitted_continuum = fit_continuum(spectrum, window=region, model=Chebyshev1D(3))
cont_fitted_region = fitted_continuum(wavs*u.angstrom)

In [ ]:
# plot your fit here to check it
plt.figure()
plt.plot(wavs, spec, color='k', lw=0.5, label='Original Spectrum')
plt.plot(wavs, cont_fitted_region, '--', color='r', lw=0.5, label='Fitted Continuum')
for i in range(len(region)):
    plt.axvspan(region[i][0].value, region[i][1].value, color='gray', lw=0.5, ls='--',alpha=0.2)
plt.xlabel('Wavelength (Angstroms)')
plt.ylabel('Flux')
plt.title('Spectrum and Fitted Continuum')
plt.legend()
plt.show()

## Compare with the correct answer

Now you have two fitted continuum, `cont_fitted` and `cont_fitted_region`.\
You are ready to do continuum normalization by dividing the original spectrum, `spec`, with your favorite one.

Compare your normalized spectrum with the correct answer, `norm_spec`, and see how each one does.

In [ ]:
# define your normalized spectrum
my_norm_spec = spec/cont_fitted_region

# plot your favorite normalized spectrum and compare it with the correct answer
plt.figure()
plt.plot(wavs, my_norm_spec, label='My Normalized Spectrum')
try:
    plt.plot(wavs, norm_spec, '--', label='Correct Normalized Spectrum')
except Exception:
    pass
plt.title(f'Normalized Spectrum - {target_name}')
plt.xlabel('Wavelength (Angstroms)')
plt.ylabel('Normalized Flux')
plt.legend()
plt.show()

You can tell that your continuum normalization is not perfect. Indeed, continuum normalization is a hard problem to solve computationally. There are many other tricks one could use to do a better job.

Once you have a continuum normalized spectrum, you are ready to calculate line characteristics such as the equivalent width.